In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

import scripts.feature_engineering as feature
from scripts.evaluate import CLIP, calculate_metrics, rmse

KeyboardInterrupt: 

In [ ]:
df = pd.read_csv('data/gold/train.csv')
df_cat = pd.read_csv('data/landing/item_categories.csv')
df_items = pd.read_csv('data/landing/items.csv')
df_shops = pd.read_csv('data/landing/shops.csv')
df_sales = pd.read_csv('data/landing/sales_train.csv')

df = feature.add_categories(df, df_cat, df_items)
df = feature.add_cities(df, df_shops)
df = feature.add_target_lags(df, (1, 2, 3, 4, 5, 6, 12))
df = feature.add_lag_stats(df)
df = feature.add_mean_lags(df)
df = feature.add_age_features(df)
df = feature.add_price_features(df, df_sales, df_items)
df = feature.add_calendar(df)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10884554 entries, 0 to 10884553
Data columns (total 34 columns):
 #   Column                Dtype  
---  ------                -----  
 0   date_block_num        int64  
 1   shop_id               int64  
 2   item_id               int64  
 3   target                float64
 4   item_category_id      uint8  
 5   category_type_id      uint8  
 6   item_type_id          uint8  
 7   city_id               uint8  
 8   target_lag_1          float32
 9   target_lag_2          float32
 10  target_lag_3          float32
 11  target_lag_4          float32
 12  target_lag_5          float32
 13  target_lag_6          float32
 14  target_lag_12         float32
 15  roll3                 float32
 16  roll6                 float32
 17  diff_1_2              float32
 18  item_mean_lag_1       float32
 19  shop_mean_lag_1       float32
 20  cat_mean_lag_1        float32
 21  shop_cat_mean_lag_1   float32
 22  city_item_mean_lag_1  float32
 23  pair_age        

## Подготовка под линейную модель

| группа | что делаем | зачем |
|---|---|---|
| счётные и скошенные (лаги, roll, средние, возраст, цена) | log1p → NaN в 0 + флаг пропуска → стандартизация | после логарифма связь с таргетом ближе к линейной; флаг отличает «пары не было в сетке» от «не продалась» |
| отношения цен | log → NaN в 0 + флаг → стандартизация | log делает отношение симметричным: вдвое дороже и вдвое дешевле равны по модулю |
| уже в нормальной шкале (diff, sin/cos, флаг новинки) | NaN в 0 → стандартизация | `diff` бывает отрицательным, log1p к нему нельзя |
| категории (магазин, категория товара) | one-hot | числовой код магазина для линейной модели ничего не значит |
| выбрасываем | `city_id`, `category_type_id`, `item_type_id`, `month`, `days` | первые три однозначно следуют из магазина и категории, месяц уже есть в sin/cos |

Преобразование обучается только на train, valid лишь трансформируется. Иначе средние и дисперсии valid попали бы в признаки.

In [ ]:
LOG1P_COLS = [
    'target_lag_1', 'target_lag_2', 'target_lag_3', 'target_lag_4', 'target_lag_5', 'target_lag_6',
    'target_lag_12', 'roll3', 'roll6',
    'item_mean_lag_1', 'shop_mean_lag_1', 'cat_mean_lag_1', 'shop_cat_mean_lag_1', 'city_item_mean_lag_1',
    'pair_age', 'pair_since_last', 'item_age', 'item_price_last',
]
LOG_COLS = ['item_price_ratio', 'price_rel_cat']
PLAIN_COLS = ['diff_1_2', 'month_sin', 'month_cos', 'item_is_new']
ONEHOT_COLS = ['shop_id', 'item_category_id']
RIDGE_FEATURES = LOG1P_COLS + LOG_COLS + PLAIN_COLS + ONEHOT_COLS

# Ridge оценивает ~200 коэффициентов: 2 млн строк хватает с запасом, а память и время падают в 3 раза.
# None = весь train.
TRAIN_SAMPLE: int | None = 2_000_000

m = df['date_block_num']
train, valid = df[(m >= 12) & (m <= 31)], df[m == 32]
if TRAIN_SAMPLE is not None and len(train) > TRAIN_SAMPLE:
    train = train.sample(TRAIN_SAMPLE, random_state=42)


def numeric(func=None):
    """func -> NaN в 0 с флагом пропуска -> стандартизация."""
    steps = [FunctionTransformer(func, feature_names_out='one-to-one')] if func else []
    return make_pipeline(
        *steps,
        SimpleImputer(strategy='constant', fill_value=0.0, add_indicator=True),
        StandardScaler(),
    )


prep = ColumnTransformer(
    [
        ('log1p', numeric(np.log1p), LOG1P_COLS),
        ('log', numeric(lambda x: np.log(np.clip(x, 1e-3, None))), LOG_COLS),
        ('plain', numeric(), PLAIN_COLS),
        ('onehot', OneHotEncoder(handle_unknown='ignore', dtype=np.float32), ONEHOT_COLS),
    ],
    sparse_threshold=1.0,  # one-hot разреженный, вся матрица остаётся разреженной
)

t = time.perf_counter()
X_train = prep.fit_transform(train[RIDGE_FEATURES])
X_valid = prep.transform(valid[RIDGE_FEATURES])
y_train, y_valid = train['target'].to_numpy(), valid['target'].to_numpy()
print(f'{time.perf_counter() - t:.0f} с, X_train {X_train.shape}, X_valid {X_valid.shape}')
print('признаков после подготовки:', len(prep.get_feature_names_out()))

7 с, X_train (2000000, 174), X_valid (213486, 174)
признаков после подготовки: 174


In [ ]:
rows = []
for alpha in [0.1, 1, 10, 100, 1_000, 10_000, 100_000]:
    t = time.perf_counter()
    ridge = Ridge(alpha=alpha).fit(X_train, y_train)
    rows.append({
        'alpha': alpha,
        'rmse_valid': rmse(y_valid, np.clip(ridge.predict(X_valid), *CLIP)),
        'fit_s': round(time.perf_counter() - t, 1),
    })
alpha_grid = pd.DataFrame(rows).set_index('alpha')
alpha_grid.round(4)

,rmse_valid,fit_s
alpha,,
0.1,0.9600,55.3
1.0,0.9600,54.5
10.0,0.9598,53.2
100.0,0.9595,49.0
1000.0,0.9623,28.1
10000.0,0.9667,11.0
100000.0,0.9720,5.1


In [ ]:
import matplotlib.pyplot as plt

# У Ridge нет итераций, поэтому вместо кривой обучения - кривая валидации: RMSE от alpha.
LGBM_VALID = 0.8526  # из lgbm/model.ipynb, для сравнения
refs = {'naive': rmse(y_valid, valid['target_lag_1'].fillna(0)), 'LightGBM': LGBM_VALID}
best = alpha_grid['rmse_valid'].idxmin()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(alpha_grid.index, alpha_grid['rmse_valid'], marker='o', label='Ridge, valid (32)')
for i, (name, value) in enumerate(refs.items(), start=1):
    ax.axhline(value, ls='--', lw=1, c=f'C{i}', label=f'{name} = {value:.3f}')
ax.axvline(best, ls=':', c='red', label=f'best alpha = {best:g}, {alpha_grid.loc[best, "rmse_valid"]:.4f}')
ax.set(xscale='log', xlabel='alpha (логарифмическая шкала)', ylabel='RMSE на valid',
       title='Ridge: RMSE от силы регуляризации')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
best_alpha = alpha_grid['rmse_valid'].idxmin()
ridge = Ridge(alpha=best_alpha).fit(X_train, y_train)
model = make_pipeline(prep, ridge)

res = calculate_metrics(
    model,
    splits={'train': (train[RIDGE_FEATURES], train['target']), 'valid': (valid[RIDGE_FEATURES], valid['target'])},
    model_name=f'Ridge (alpha={best_alpha:g})',
    baselines={'zeros': np.zeros(len(valid)), 'naive': valid['target_lag_1'].fillna(0)},
    backend='plotly'
)
res.round(4)

,rmse,mae,bias,r2,rmse_nonzero
train,0.9452,0.3490,0.0394,0.3752,2.2798
valid,0.9595,0.3540,0.0639,0.3244,2.3542
valid: zeros,1.1982,0.2701,-0.2701,-0.0536,3.2177
valid: naive,1.0766,0.3202,0.0106,0.1494,2.6185
